In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from torch.utils.data import TensorDataset, DataLoader

In [2]:
# generate the same synthetic dataset as before
np.random.seed(42)

n_samples = 1000
d = 2

mu_0 = np.array([-2.0, 0.0])
mu_1 = np.array([2.0, 0.0])

sigma = 1.0

y = np.random.randint(0, 2, size=n_samples)

X = np.zeros((n_samples, d))

for i in range(n_samples):
    if y[i] == 0:
        X[i] = np.random.normal(mu_0, sigma, d)
    else:
        X[i] = np.random.normal(mu_1, sigma, d)

In [3]:
# find 10 nearest neighbors for each point
k = 10

knn = NearestNeighbors(n_neighbors=k + 1)
knn.fit(X)

distances, indices = knn.kneighbors(X)

neighbor_indices = indices[:, 1:]

In [4]:
# apply label modification
neighbor_label_mean = y[neighbor_indices].mean(axis=1)

epsilon = 0.1

ambiguous = np.abs(
    neighbor_label_mean - 0.5
) < epsilon

y_modified = y.copy()

y_modified[ambiguous] = 1 - y_modified[ambiguous]

print("Number of flipped labels:", ambiguous.sum())
print("Percentage flipped:", ambiguous.mean() * 100)

Number of flipped labels: 22
Percentage flipped: 2.1999999999999997


In [5]:
# center and neighbor data
X_center = X
X_neighbors = X[neighbor_indices]

print("Center shape:", X_center.shape)
print("Neighbors shape:", X_neighbors.shape)
print("Labels shape:", y_modified.shape)

Center shape: (1000, 2)
Neighbors shape: (1000, 10, 2)
Labels shape: (1000,)


In [6]:
# train-test split
X_center_train, X_center_test, X_neighbors_train, X_neighbors_test, y_train, y_test = train_test_split(
    X_center,
    X_neighbors,
    y_modified,
    test_size=0.2,
    random_state=42,
    stratify=y_modified
)

# convert to PyTorch tensors
X_center_train = torch.tensor(
    X_center_train,
    dtype=torch.float32
)

X_center_test = torch.tensor(
    X_center_test,
    dtype=torch.float32
)

X_neighbors_train = torch.tensor(
    X_neighbors_train,
    dtype=torch.float32
)

X_neighbors_test = torch.tensor(
    X_neighbors_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
# dataloaders
train_dataset = TensorDataset(
    X_center_train,
    X_neighbors_train,
    y_train
)

test_dataset = TensorDataset(
    X_center_test,
    X_neighbors_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [8]:
# define PointNet local features
class PointNetLocal(nn.Module):

    def __init__(self, input_dim=2, feature_dim=32):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, feature_dim),
            nn.ReLU()
        )

    def forward(self, neighbors):

        features = self.mlp(neighbors)

        pooled = torch.max(
            features,
            dim=1
        ).values

        return pooled

In [9]:
# test local PointNet
local_pointnet = PointNetLocal(
    input_dim=2,
    feature_dim=32
)

center_batch = X_center_train[:32]
neighbors_batch = X_neighbors_train[:32]

local_features = local_pointnet(
    neighbors_batch
)

print("Input neighbors shape:", neighbors_batch.shape)
print("Local features shape:", local_features.shape)

Input neighbors shape: torch.Size([32, 10, 2])
Local features shape: torch.Size([32, 32])


In [10]:
# define hierarchical PointNet block
class PointNetPlusPlus(nn.Module):

    def __init__(self):
        super().__init__()

        self.local_pointnet = PointNetLocal(
            input_dim=2,
            feature_dim=32
        )

        self.second_layer = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(32 + 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, center, neighbors):

        local_features = self.local_pointnet(
            neighbors
        )

        hierarchical_features = self.second_layer(
            local_features
        )

        combined = torch.cat(
            [center, hierarchical_features],
            dim=1
        )

        output = self.classifier(
            combined
        )

        return output.squeeze(1)

In [11]:
# instantiate and test
model = PointNetPlusPlus()

output = model(
    center_batch,
    neighbors_batch
)

print("Output shape:", output.shape)

Output shape: torch.Size([32])


In [12]:
# loss and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [13]:
# train model for 50 epochs
num_epochs = 50

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for center_batch, neighbors_batch, labels_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(
            center_batch,
            neighbors_batch
        )

        loss = criterion(
            outputs,
            labels_batch
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}: "
            f"Loss = {average_loss:.4f}"
        )

Epoch 10: Loss = 0.0639
Epoch 20: Loss = 0.0481
Epoch 30: Loss = 0.0439
Epoch 40: Loss = 0.0435
Epoch 50: Loss = 0.0357


In [14]:
# evaluate model
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for center_batch, neighbors_batch, labels_batch in test_loader:

        outputs = model(
            center_batch,
            neighbors_batch
        )

        probabilities = torch.sigmoid(outputs)

        predictions = (
            probabilities >= 0.5
        ).float()

        all_predictions.extend(
            predictions.numpy()
        )

        all_labels.extend(
            labels_batch.numpy()
        )

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Class 0", "Class 1"]
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)


Accuracy: 0.9600

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.97      0.95      0.96        98
     Class 1       0.95      0.97      0.96       102

    accuracy                           0.96       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.96      0.96      0.96       200

Confusion Matrix:
[[93  5]
 [ 3 99]]


In [15]:
# check permutation invariance
model.eval()

center = X_center_test[0:1]
neighbors = X_neighbors_test[0:1]

with torch.no_grad():

    original_probability = torch.sigmoid(
        model(center, neighbors)
    ).item()

differences = []

for _ in range(10):

    permutation = torch.randperm(
        neighbors.shape[1]
    )

    shuffled_neighbors = neighbors[
        :,
        permutation,
        :
    ]

    with torch.no_grad():

        shuffled_probability = torch.sigmoid(
            model(
                center,
                shuffled_neighbors
            )
        ).item()

    differences.append(
        abs(
            original_probability
            - shuffled_probability
        )
    )

print(
    "Original probability:",
    original_probability
)

print(
    "Maximum difference:",
    max(differences)
)

print(
    "All differences:",
    differences
)

Original probability: 0.9999679327011108
Maximum difference: 0.0
All differences: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
